# 02 · QLoRA:原理与实战

> 前置:请先完成 [`00`](00_环境准备与GPU检测.ipynb) 的环境检测(尤其是 4-bit 量化测试通过),并建议先读完 [`01`](01_PEFT与LoRA原理与实战.ipynb) 了解 LoRA。

**QLoRA = 量化 (Quantization) + LoRA。**

`01` 里 LoRA 已经省下了「训练参数量」,但底座仍以 bf16 加载(7B 约 14GB)。
QLoRA 更进一步:**把冻结的底座用 4-bit 加载**,显存直接砍到约 1/4,于是 16GB 的 5080 也能微调 7B。

本 notebook:讲清量化和 NF4 的原理,然后用 4-bit 加载 `Qwen2.5-7B-Instruct` 并做 QLoRA 微调。

## 一、量化基础

模型参数默认用 16 位浮点(fp16/bf16)存,每个数占 2 字节。**量化**就是用更少的位数来近似表示这些数,比如用 4 位(0.5 字节),显存立刻降到 1/4。

代价是精度损失,但对**推理和「冻结的底座」**来说,这点损失通常可以接受。核心做法:

- 把一组权重的取值范围,映射到少数几个「量化档位」上。
- 记录一个缩放因子(scale),用时再反量化回近似的浮点值。

QLoRA 论文(2023)提出了三个关键技术,让 4-bit 量化几乎不掉点:

## 二、QLoRA 的三个关键点

1. **NF4 (4-bit NormalFloat)**
   神经网络权重近似服从正态分布。NF4 是一种「为正态分布量身定制」的 4-bit 数据类型,它的 16 个量化档位在正态分布下是**信息论最优**的,比普通的 4-bit 整数量化更准。

2. **Double Quantization(双重量化)**
   量化本身会产生很多「缩放因子(scale)」,这些 scale 也占显存。双重量化就是**把这些 scale 再量化一次**,进一步省下约 0.4 bit/参数。

3. **Compute dtype(计算精度)**
   权重以 4-bit **存储**,但真正参与矩阵乘法时会**临时反量化**成 bf16/fp16 来算(`bnb_4bit_compute_dtype`),保证计算精度。

**QLoRA 的完整图景:**

```
        4-bit NF4 冻结底座  ←── 省显存,只用于前向,不更新
                +
        bf16 的 LoRA 适配器  ←── 唯一被训练的部分
```

梯度只流过 LoRA 适配器,4-bit 底座全程冻结。所以 QLoRA 既省「存储显存」(4-bit 底座),又省「训练显存」(只有 LoRA 有梯度和优化器状态)。

## 三、4-bit 加载 Qwen2.5-7B

关键就是 `BitsAndBytesConfig`。首次运行会下载 7B 模型(约 15GB),**下载较久**。

> 如果不想下 7B(磁盘/网络紧张),把下面的 `MODEL_NAME` 换成 `Qwen/Qwen2.5-3B-Instruct` 或 `1.5B`,流程完全一样。

In [ ]:
import os
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"  # 下载慢可开启镜像

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"   # 显存/磁盘紧张可改为 Qwen2.5-3B-Instruct
DATA_PATH = "../data/instructions.jsonl"
OUTPUT_DIR = "../outputs/qlora-qwen7b"
MAX_LEN = 512

# ★ QLoRA 的核心:4-bit 量化配置
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 用 NF4 而非普通 int4
    bnb_4bit_use_double_quant=True,         # 双重量化,再省一点
    bnb_4bit_compute_dtype=torch.bfloat16,  # 计算时临时反量化成 bf16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
)
model.config.use_cache = False

print(f"已用 4-bit 加载: {MODEL_NAME}")
print(f"参数总量: {model.num_parameters()/1e9:.2f} B")
print(f"加载后显存占用: {torch.cuda.memory_allocated()/1024**3:.2f} GB "
      f"(7B 若用 bf16 加载约需 14GB,这里 4-bit 只要约 1/4)")

## 四、为 k-bit 训练做准备 + 挂 LoRA

对量化模型微调,要先调用 `prepare_model_for_kbit_training`:它会把 LayerNorm 等留在 fp32、开启梯度检查点、处理输入梯度等,保证训练稳定。之后挂 LoRA 的写法和 `01` 完全一样。

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 为 4-bit 训练做准备(梯度检查点省显存)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 五、准备数据(与 01 相同的处理)

数据处理逻辑和 `01` 一致:套 chat template、只对 assistant 回答算 loss。这里重写一遍让本 notebook 可独立运行。

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = "你是一个热心的中文学习小助手。"
raw = load_dataset("json", data_files=DATA_PATH, split="train")


def build_example(sample):
    user_content = sample["instruction"]
    if sample.get("input"):
        user_content += "\n" + sample["input"]
    prefix_ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_content}],
        tokenize=True, add_generation_prompt=True,
    )
    full_ids = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user_content},
         {"role": "assistant", "content": sample["output"]}],
        tokenize=True, add_generation_prompt=False,
    )[:MAX_LEN]
    labels = list(full_ids)
    for i in range(min(len(prefix_ids), len(labels))):
        labels[i] = -100
    return {"input_ids": full_ids, "labels": labels,
            "attention_mask": [1] * len(full_ids)}


dataset = raw.map(build_example, remove_columns=raw.column_names)
print(f"处理完成,样本数: {len(dataset)}")

## 六、训练

QLoRA 训练常用 `paged_adamw_8bit` 优化器(8-bit 分页优化器,进一步省显存)。同样把 `max_steps` 设小做演示。

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

torch.cuda.reset_peak_memory_stats()
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    max_steps=60,                       # 演示用,出效果请调大
    logging_steps=5,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    bf16=True,
    optim="paged_adamw_8bit",           # QLoRA 常用的省显存优化器
    gradient_checkpointing=True,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(model=model, args=args,
                  train_dataset=dataset, data_collator=collator)
trainer.train()
print(f"\nQLoRA 训练显存峰值: {torch.cuda.max_memory_allocated()/1024**3:.2f} GB "
      f"(能在 16GB 上微调 7B!)")

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"QLoRA 适配器已保存到: {OUTPUT_DIR}")

## 小结:LoRA vs QLoRA

| 对比项 | LoRA (01) | QLoRA (02) |
| --- | --- | --- |
| 底座加载精度 | bf16 (16-bit) | NF4 (4-bit) |
| 底座显存 (7B) | ~14GB | ~4-5GB |
| 可训练参数 | LoRA 适配器 | LoRA 适配器(相同) |
| 训练速度 | 较快 | 略慢(反量化开销) |
| 精度损失 | 无 | 极小(NF4 设计得好) |
| 适用场景 | 显存够、追求速度 | 显存紧张、要训大模型 |

一句话:**QLoRA 用一点点速度和精度,换来了在小显存上微调大模型的能力。**

关于 QLoRA 的几个易错点:
- 必须 `prepare_model_for_kbit_training`,否则梯度可能不通。
- 4-bit 量化模型**不要**用 `merge_and_unload` 直接合并后当高精度模型用(会掉点);要合并请先用 fp16 重新加载底座再合并(见 `03`)。

接下来:[`03_微调后_合并推理与评估`](03_微调后_合并推理与评估.ipynb),看看微调到底有没有效果。